# Tugas 2: Pantau model

Di <i>notebook </i>ini, Anda akan memantau dan mengevaluasi data yang diambil dari titik akhir. Anda harus membuat <i>baseline </i>yang akan Anda bandingkan dengan lalu lintas <i>real-time</i>. Setelah <i>baseline </i>siap, Anda harus mengatur jadwal untuk terus mengevaluasi dan membandingkan data dengan <i>baseline </i>tersebut.

## Tugas 2.1: Atur lingkungan

Dalam tugas ini, Anda akan mengatur lingkungan.

In [ ]:
#install-dependencies
%matplotlib inline
from datetime import datetime, timedelta
import json
import boto3
import time
import pandas as pd
import matplotlib.pyplot as plt
from sagemaker import get_execution_role, session
from sagemaker.s3 import S3Uploader
from sagemaker.image_uris import retrieve
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
from time import sleep
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

region = boto3.Session().region_name
role = get_execution_role()
sm_session = session.Session(boto3.Session())
sm = boto3.Session().client("sagemaker")
sm_runtime = boto3.Session().client("sagemaker-runtime")
cw = boto3.Session().client("cloudwatch")

bucket = sm_session.default_bucket()
prefix = 'sagemaker/abalone'
data_capture_prefix = "{}/datacapture".format(prefix)
s3_capture_upload_path = "s3://{}/{}".format(bucket, data_capture_prefix)
capture_modes = [ "Input",  "Output" ]
code_prefix = "{}/code".format(prefix)
s3_code_preprocessor_uri = "s3://{}/{}/{}".format(bucket, code_prefix, "preprocessor.py")
s3_code_postprocessor_uri = "s3://{}/{}/{}".format(bucket, code_prefix, "postprocessor.py")
reports_prefix = "{}/reports".format(prefix)
s3_report_path = "s3://{}/{}".format(bucket, reports_prefix)


## Tugas 2.2: Buat titik akhir produksi dengan mengaktifkan Data Capture

Untuk mencatat input ke titik akhir dan <i>output </i>inferensi dari model yang Anda gunakan ke Amazon S3, Anda dapat mengaktifkan fitur yang disebut Data Capture. Data Capture mencatat informasi yang dapat digunakan untuk pelatihan, <i>debugging</i>, dan <i>monitoring</i>. Amazon SageMaker Model Monitor secara otomatis mengurai data yang diambil ini dan membandingkan metrik darinya dengan <i>baseline </i>model yang Anda buat.

Dalam tugas ini, Anda akan mengunggah model yang telah terlatih ke <i>bucket </i>S3, membuat objek model Amazon Sagemaker, mengonfigurasi titik akhir <i>real-time</i> Amazon SageMaker dengan mengaktifkan Data Capture, dan membuat titik akhir <i>real-time</i>.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Catatan:** Pembuatan titik akhir akan selesai dalam waktu sekitar 5 menit.

In [ ]:
#create-production-endpoint
# Upload models
model_url = S3Uploader.upload(
    local_path="models/model.tar.gz", desired_s3_uri=f"s3://{bucket}/{prefix}"
)

# Create the model definitions
model_name = f"abalone-A-{datetime.now():%Y-%m-%d-%H-%M-%S}"
image_uri = retrieve("xgboost", boto3.Session().region_name, "1.5-1")

# Create production model object
predictor=sm_session.create_model(
    name=model_name, role=role, container_defs={"Image": image_uri, "ModelDataUrl": model_url}
)

# Create the endpoint configurations
variant_name = 'AllTraffic'

endpoint_config_name = f'Abalone-Endpoint-1-{datetime.now():%Y-%m-%d-%H-%M-%S}'
endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName = endpoint_config_name,
    ProductionVariants=[
        {
            'ModelName':model_name,
            'InstanceType':'ml.m5.xlarge',
            'InitialInstanceCount':1,
            'VariantName':variant_name
        }
    ],
    
        DataCaptureConfig= {
        'EnableCapture': True, # Whether data should be captured or not.
        'InitialSamplingPercentage' : 100,
        'CaptureContentTypeHeader': {'CsvContentTypes': [ 'text/csv' ]},
        'DestinationS3Uri': s3_capture_upload_path,
        'CaptureOptions': [{"CaptureMode" : capture_mode} for capture_mode in capture_modes] # Example - Use list comprehension to capture both Input and Output
    }
)
print(f"Created the Production Model Endpoint Config: {endpoint_config_name}")
time.sleep(5)

# Create the endpoint with the production model
endpoint_name = f"Abalone-{datetime.now():%Y-%m-%d-%H-%M-%S}"
endpoint_response = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)

def wait_for_endpoint_creation_complete(endpoint):
    """Helper function to wait for the completion of creating an endpoint"""
    response = sm.describe_endpoint(EndpointName=endpoint_name)
    status = response.get("EndpointStatus")
    while status == "Creating":
        print("Waiting for Endpoint Creation")
        time.sleep(15)
        response = sm.describe_endpoint(EndpointName=endpoint_name)
        status = response.get("EndpointStatus")

    if status != "InService":
        print(f"Failed to create endpoint, response: {response}")
        failureReason = response.get("FailureReason", "")
        raise SystemExit(
            f"Failed to create endpoint {endpoint_response['EndpointArn']}, status: {status}, reason: {failureReason}"
        )
    print(f"Endpoint {endpoint_response['EndpointArn']} successfully created.")

wait_for_endpoint_creation_complete(endpoint=endpoint_response)

Ketika sel selesai, akan muncul titik akhir ARN *arn:aws:sagemaker:us-west-2:012345678910:endpoint/abalone-2040-10-11-10-11-12*.

Titik akhir Anda saat ini dikonfigurasi dengan satu varian, yaitu model produksi. Anda dapat melihat konfigurasi titik akhir menggunakan *describe_endpoint*.

In [ ]:
#describe-the-endpoint
sm.describe_endpoint(EndpointName=endpoint_name)

## Tugas 2.3: Lihat data yang diambil

Dalam tugas ini, Anda akan memanggil titik akhir yang telah dibuat di atas menggunakan data produksi. Karena Data Capture di titik akhir sudah diaktifkan, <i>payload </i>permintaan, respons, dan metadata tambahan disimpan di lokasi S3 yang Anda tentukan sebelumnya di <i>notebook</i>. Setelah memanggil titik akhir, Anda harus memeriksa data yang diambil di <i>bucket </i>S3.

Pertama, gunakan prediktor yang diinisialisasi dan dikonfigurasi dengan nama titik akhir untuk memanggil titik akhir tersebut. Pemanggilan titik akhir dengan catatan baru membantu Anda memastikan bahwa pengaturan konfigurasi Data Capture sudah benar. Prediktor akan membuat permintaan prediksi ke titik akhir Amazon SageMaker. Kemudian, jalankan inferensi dengan mengirim catatan ke titik akhir.

In [ ]:
#invoke-the-endpoint
predictor = Predictor(endpoint_name=endpoint_name,
                        serializer=CSVSerializer(),
                        deserializer=CSVDeserializer())

validate_dataset = "abalone_data_new_predictions.csv"

limit = 200  # Need at least 200 samples to compute standard deviations
i = 0
with open(f"data/{validate_dataset}", "w") as validation_file:
    validation_file.write("prediction,label\n")  # CSV header
    with open("data/abalone_data_new.csv", "r") as f:
        for row in f:
            (label, input_cols) = row.split(",", 1)
            prediction = predictor.predict(input_cols)[0][0]
            validation_file.write(f"{prediction},{label}\n")
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            sleep(0.5)

print("\nDone!")

Selanjutnya, cantumkan <i>file </i>Data Capture yang disimpan di Amazon S3.

In [ ]:
#list-data-capture-files
s3_client = boto3.Session().client("s3")
current_endpoint_capture_prefix = "{}/{}".format(data_capture_prefix, endpoint_name)
result = s3_client.list_objects(Bucket=bucket, Prefix=current_endpoint_capture_prefix)
capture_files = [capture_file.get("Key") for capture_file in result.get("Contents")]
print("Found Capture Files:")
print("\n ".join(capture_files))

Kemudian, lihat isi dari salah satu <i>file </i>Data Capture. Anda harus melihat semua data yang diambil dalam <i>file </i>berformat JSON-line khusus Amazon SageMaker. Luangkan waktu sejenak untuk meninjau beberapa baris pertama dalam <i>file </i>yang diambil.

In [ ]:
#view-captured-file-lines
def get_obj_body(obj_key):
    return s3_client.get_object(Bucket=bucket, Key=obj_key).get("Body").read().decode("utf-8")


capture_file = get_obj_body(capture_files[-1])
print(capture_file[:2000])

Terakhir, lihat isi dari salah satu catatan input dan <i>output </i>titik akhir yang diambil.

In [ ]:
#print-json-file
print(json.dumps(json.loads(capture_file.split("\n")[0]), indent=2))

Dengan mengaktifkan Data Capture pada titik akhir, penyimpanan informasi pelatihan, <i>debugging</i>, dan <i>monitoring </i>menjadi lebih fleksibel. Karena Amazon SageMaker Model Monitor mengurai data yang diambil ini secara otomatis, fitur Data Capture membantu Anda membandingkan catatan baru dengan data <i>baseline</i>. 

Anda belum mengonfigurasi <i>baseline</i>. Pada tugas berikutnya, Anda akan menggunakan SageMaker Model Monitor untuk menghasilkan statistik <i>baseline </i>dan batasan. 

## Tugas 2.4: Hasilkan statistik <i>baseline </i>dan batasan

Dalam tugas ini, Anda akan membuat <i>baseline</i>. Statistik <i>baseline </i>dan batasan berfungsi sebagai standar untuk mendeteksi penyimpangan data dan masalah kualitas data lainnya. 

Set data pengujian dari pelatihan model sering kali berkualitas baik. Skema set data pengujian dan inferensi harus sama persis, termasuk jumlah dan urutan fiturnya. Dari set data pengujian, Anda dapat meminta Amazon SageMaker untuk menyarankan serangkaian batasan <i>baseline </i>dan menghasilkan statistik deskriptif untuk menganalisis data.

Pertama, konfigurasikan prefiks <i>baseline </i>dan variabel.

In [ ]:
#configure-baseline-variables
baseline_prefix = prefix + "/baselining"
baseline_data_prefix = baseline_prefix + "/data"
baseline_results_prefix = baseline_prefix + "/results"

baseline_data_uri = "s3://{}/{}".format(bucket, baseline_data_prefix)
baseline_results_uri = "s3://{}/{}".format(bucket, baseline_results_prefix)
print("Baseline data uri: {}".format(baseline_data_uri))
print("Baseline results uri: {}".format(baseline_results_uri))

Selanjutnya, mulailah tugas untuk menyarankan <i>baseline</i> dan batasan. *DefaultModelMonitor.suggest_baseline()* akan memulai **ProcessingJob** menggunakan kontainer Model Monitor yang disediakan Amazon SageMaker untuk menghasilkan <i>baseline </i>dan batasan.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Catatan:** Tugas <i>baseline </i>akan selesai dalam waktu sekitar 10 menit.

<i aria-hidden="true" class="fas fa-info-circle" style="color:#007FAA"></i> **Pelajari selengkapnya:** Lihat [Create a Baseline] (https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-create-baseline.html) untuk informasi selengkapnya tentang penghitungan statistik <i>baseline </i>dan batasan.

In [ ]:
#create-baselining-job
my_default_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

my_default_monitor.suggest_baseline(
    baseline_dataset="data/abalone_data_new_withheader.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    wait=True,
)

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Catatan:** Kode ini menghasilkan respons yang panjang. Anda dapat mengabaikan semua peringatan atau pesan kesalahan.

Ketika sel selesai, akan muncul sebuah pesan *2025-10-11 12:13:14,156 - DefaultDataAnalyzer - INFO - Spark job completed*.

Sekarang, cari lokasi penyimpanan <i>file </i>*constraints.json* dan *statistics.json*.

In [ ]:
#explore-generated-constraints-and-statistics
s3_client = boto3.Session().client("s3")
result = s3_client.list_objects(Bucket=bucket, Prefix=baseline_results_prefix)
report_files = [report_file.get("Key") for report_file in result.get("Contents")]
print("Found Files:")
print("\n ".join(report_files))

Selanjutnya, lihat statistik yang dihasilkan.

In [ ]:
#view-statistics
baseline_job = my_default_monitor.latest_baselining_job
schema_df = pd.json_normalize(baseline_job.baseline_statistics().body_dict["features"])
schema_df.head(10)

Tabel statistik menunjukkan setiap fitur dengan statistik ringkas yang sesuai, termasuk rata-rata, deviasi standar, nilai minimum, nilai maksimum, dan detail penting lainnya.

Terakhir, lihat batasan yang dihasilkan.

In [ ]:
#view-constraints
constraints_df = pd.json_normalize(
    baseline_job.suggested_constraints().body_dict["features"]
)
constraints_df.head(10)

Tabel batasan menunjukkan jenis yang disimpulkan dari setiap fitur, kelengkapan catatan (dalam hal ini 1.0 untuk semua fitur karena nilai <i>file </i>lengkap), dan kolom yang memiliki nilai negatif. 

Setelah Anda membuat <i>baseline </i>dan melihat statistik dan batasan, buat tugas pemantauan kualitas data Model Monitor untuk melacak catatan inferensi baru terhadap <i>baseline</i>.

## Tugas 2.5: Buat tugas pemantauan kualitas data Model Monitor

Setelah membuat <i>baseline</i>, Anda dapat menggunakan metode *create_monitoring_schedule()* dari instans kelas *DefaultModelMonitor* untuk menjadwalkan pemantauan kualitas data per jam.

Dalam tugas ini, Anda akan menganalisis dan memantau data dengan tugas pemantauan kualitas data.

Pertama, gunakan metode *create_monitoring_schedule()* untuk menjadwalkan pemantauan kualitas data per jam. 

In [ ]:
#create-monitoring-schedule
bucket = boto3.Session().resource("s3").Bucket(bucket)
bucket.Object(code_prefix + "/postprocessor.py").upload_file("python/postprocessor.py")

mon_schedule_name = f"model-monitor-schedule-{datetime.now():%Y-%m-%d-%H-%M-%S}"

my_default_monitor.create_monitoring_schedule(
    monitor_schedule_name=mon_schedule_name,
    endpoint_input=predictor.endpoint_name,
    post_analytics_processor_script=s3_code_postprocessor_uri,
    output_s3_uri=s3_report_path,
    statistics=my_default_monitor.baseline_statistics(),
    constraints=my_default_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

Selanjutnya, kirim beberapa lalu lintas buatan ke titik akhir agar tugas pemantauan dapat menghasilkan laporan pelanggaran. Untuk menyimulasikan penyimpangan data, gunakan satu set data yang tidak seimbang. Jika data tersebut dibandingkan dengan <i>baseline</i>, sistem pemicu peringatan otomatis akan mengirimkan peringatan.

In [ ]:
#send-artificial-traffic
endpoint_name = predictor.endpoint_name
runtime_client = sm_session.sagemaker_runtime_client
limit = 200
i = 0

# repeating code from above to run this section independently
def invoke_endpoint(ep_name, file_name, runtime_client):
    i = 0
    with open(file_name, "r") as f:
        for row in f:
            (label, payload) = row.strip("\n").split(",", 1)  

            response = runtime_client.invoke_endpoint(
                EndpointName=ep_name, ContentType="text/csv", Body=payload
            )
            response["Body"].read()
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            time.sleep(0.5)


invoke_endpoint(endpoint_name, "data/abalone_data_skewed.csv", runtime_client)
print("\nDone!")

Gunakan *describe_schedule* untuk melihat jadwal yang baru saja Anda buat.

In [ ]:
#model-monitor-schedule-status
desc_schedule_result = my_default_monitor.describe_schedule()
print("Schedule status: {}".format(desc_schedule_result["MonitoringScheduleStatus"]))

Jadwal monitor memulai tugas pada interval jam yang ditentukan sebelumnya. Bahkan untuk jadwal per jam, Amazon SageMaker memiliki periode penyangga selama 20 menit untuk menjadwalkan eksekusi Anda. Anda mungkin akan melihat eksekusi dimulai dari 0 hingga 20 menit setelah batasan jam. Ini adalah hal yang normal dan dilakukan untuk <i>load balancing</i> di <i>backend</i>.

Eksekusi ini memakan waktu sekitar satu jam untuk menghasilkan laporan pelanggaran. Untuk tujuan lab ini, sel berikutnya memiliki cuplikan kode yang bisa Anda lihat dan memberikan <i>output </i>sampel untuk referensi. Pada langkah terakhir tugas ini, Anda dapat melihat laporan pelanggaran dari <i>file </i>yang telah dibuat dan dimuat dari proses pemantauan sebelumnya.

Ketika eksekusi selesai, SageMaker melaporkan status eksekusi terakhir yang selesai atau gagal. 

Berikut adalah kemungkinan status terminal:
- **Completed** - Eksekusi pemantauan selesai dan tidak ada masalah yang ditemukan dalam laporan pelanggaran. 
- **CompledWithViolations** - Eksekusi selesai, tetapi terdeteksi adanya pelanggaran batasan. 
- **Failed** - Eksekusi pemantauan gagal, mungkin karena kesalahan klien (mungkin izin <i>role </i>yang salah) atau masalah infrastruktur. Pemeriksaan lebih lanjut pada FailureReason dan ExitMessage perlu dilakukan untuk mengidentifikasi apa yang sebenarnya terjadi. 
- **Stopped** - Tugas melebihi <i>runtime </i>maksimum atau dihentikan secara manual.


Jika ingin membuat daftar dan melihat status eksekusi saat ini, Anda dapat menggunakan kode seperti ini:

```python 
# list the current execution
mon_executions = my_default_monitor.list_executions()
print(
    "We created a hourly schedule above that begins executions ON the hour (plus 0-20 min buffer.\nWe will have to wait for an hour..."
)

while len(mon_executions) == 0:
    print("Waiting for the first execution to happen...")
    time.sleep(60)
    mon_executions = my_default_monitor.list_executions()
```



```python
# Latest execution status
latest_execution = mon_executions[-1] # Latest execution's index is -1, second to last is -2, etc
time.sleep(60)
latest_execution.wait(logs=False)

print("Latest execution status: {}".format(latest_execution.describe()["ProcessingJobStatus"]))
print("Latest execution result: {}".format(latest_execution.describe()["ExitMessage"]))

latest_job = latest_execution.describe()
if latest_job["ProcessingJobStatus"] != "Completed":
    print(
        "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
    )
```

The following is the expected output when the latest execution of the monitoring job completes.

```bash
!Latest execution status: Completed

Latest execution result: CompletedWithViolations: Job completed successfully with 8 violations.
```

Untuk mendata laporan pelanggaran yang dihasilkan, Anda dapat menggunakan kode seperti ini:

```python

from urllib.parse import urlparse

report_uri = latest_execution.output.destination
s3uri = urlparse(report_uri)
report_bucket = s3uri.netloc
report_key = s3uri.path.lstrip("/")
s3_client = boto3.Session().client("s3")
result = s3_client.list_objects(Bucket=report_bucket, Prefix=report_key)
report_files = [report_file.get("Key") for report_file in result.get("Contents")]
print("Found Report Files:")
print("\n ".join(report_files))
```

Berikut ini adalah daftar <i>file </i>laporan.

```bash
Found Report Files:
sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/constraint_violations.json

sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/constraints.json

sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/statistics.json
```

Untuk mendata pelanggaran perbandingan <i>baseline</i>, Anda dapat menggunakan kode seperti ini:

```python
violations = my_default_monitor.latest_monitoring_constraint_violations()
pd.set_option("display.max_colwidth", None)
constraints_df = pd.json_normalize(violations.body_dict["violations"])
constraints_df.head(10)
```

Karena eksekusi tugas pemantauan yang Anda mulai di atas tidak akan selesai dalam 60—80 menit, lihat laporan pelanggaran <i>file </i>yang telah dibuat dan dimuat dari proses pemantauan sebelumnya.

In [ ]:
#print-violations-report
pd.set_option('display.max_colwidth', None)
violations = json.load(open('data/violations.json'))
constraints_df=pd.json_normalize(violations, record_path=['violations'])
constraints_df.head(10)

Laporan tersebut menunjukkan delapan pelanggaran dalam <i>file </i>yang tidak seimbang, dengan empat pelanggaran **data_type_check** untuk **rings**, **sex_f**, **sex_i**, dan **sex_m**, dan empat pelanggaran **baseline_drift_check** untuk **length**, **diameter**, **whole_weight**, dan **shell_weight**.

Luangkan waktu sejenak untuk melihat deskripsi setiap pelanggaran. Perhatikan bahwa kecocokan tipe data tidak benar untuk beberapa fitur. Perhatikan juga bahwa jarak penyimpangan <i>baseline </i>melebihi ambang batas *0,1* untuk empat fitur yang dilaporkan.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Catatan:** Saat Anda membuat tugas <i>baseline</i>, proses tersebut akan menghasilkan <i>file </i>constraints.json dan statistics.json. Dalam <i>file </i>*constraints.json*, *comparison_threshold* ditetapkan ke *0,1* secara <i>default</i>. Untuk mempelajari selanjutnya tentang <i>file </i>constraints.json, lihat [Schema for Constraints](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-byoc-constraints.html).

## Tugas 2.6: Buat alarm CloudWatch

Ketika terjadi penyimpangan data, alarm ini sangat membantu untuk mengirimkan notifikasi sehingga Anda dapat mengatasi masalah apa pun. Notifikasi atau alarm juga dapat memicu pelatihan ulang model otomatis untuk mengatasi perubahan yang mungkin terjadi pada data inferensi Anda.

Dalam tugas ini, Anda akan mempelajari cara membuat alarm dan mengaktifkan notifikasi untuk mengetahui saat data menyimpang dari data <i>baseline</i>.

Pertama, temukan topik Amazon Simple Notification Service (Amazon SNS) Anda saat ini dengan menggunakan**list_topics**.

In [ ]:
#list-topics
client = boto3.client('sns')
topics_list= client.list_topics()
print(topics_list)

Selanjutnya, atur variabel **sns_notifications_topic** dengan nilai ARN topik yang Anda temukan di sel sebelumnya.

In [ ]:
#set-variables
topic_details = pd.json_normalize(topics_list['Topics'])
topic_arn = topic_details['TopicArn']
print (topic_arn[0])
sns_notifications_topic = topic_arn[0]

Terakhir, buat alarm menggunakan **put_metric_alarm** yang memicu notifikasi ketika diameter fitur berubah dari <i>baseline </i>dan melatih ulang model secara otomatis. 

Anda dapat menggunakan kontainer Amazon SageMaker Model Monitor bawaan untuk metrik CloudWatch. SageMaker memancarkan metrik untuk setiap fitur yang diamati dalam set data di <i>namespace </i>*/aws/sagemaker/Endpoints/data-metric* dengan dimensi *EndpointName* dan *ScheduleName*

In [ ]:
#trigger-cloudwatch-alarm-when-it-drifts-from-baseline
previous_date = f'{datetime.today() - timedelta(days=1):%Y-%m-%d}'
print (previous_date)
cw_client = boto3.Session().client('cloudwatch')

alarm_name = 'BASELINE_DRIFT_FEATURE_DIAMETER'
alarm_desc = 'Trigger a cloudwatch alarm when the feature diameter drifts away from the baseline'
feature_diameter_drift_threshold = 0.1  # Setting this threshold purposefully low to see the alarm quickly.
metric_name = 'feature_baseline_drift_diameter'
namespace = 'aws/sagemaker/Endpoints/data-metrics'

endpoint_name = 'Abalone-' + previous_date
print (endpoint_name)
monitoring_schedule_name = 'model-monitor-schedule-' + previous_date
print (monitoring_schedule_name)

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    AlarmActions=[sns_notifications_topic],
    MetricName=metric_name,
    Namespace=namespace,
    Statistic='Sum',
    Dimensions=[
        {
            'Name': 'Endpoint',
            'Value': endpoint_name
        },
        {
            'Name': 'MonitoringSchedule',
            'Value': monitoring_schedule_name
        }
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=feature_diameter_drift_threshold,
    ComparisonOperator='GreaterThanOrEqualToThreshold',
    TreatMissingData='breaching'
)

Anda berhasil membuat alarm CloudWatch. Anda dapat menggunakan alarm ini untuk mengetahui adanya masalah penyimpangan data dan memicu pelatihan ulang model otomatis.

### Pembersihan

Anda telah menyelesaikan <i>notebook </i>ini. Untuk pindah ke bagian lab berikutnya, lakukan hal berikut:

- Tutup <i>file notebook</i> ini.
- Kembali ke sesi lab dan lanjutkan dengan **Tugas 3**.